# Student Performance Analysis

This notebook analyzes a clearly labelled synthetic student performance dataset using Pandas and Matplotlib.

**Note:** The dataset is synthetic and for demonstration/portfolio purposes only. Do not present findings as real-world facts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Resolve project root whether notebook is run from repo root or notebooks/
cwd = Path.cwd()
root = cwd if (cwd / 'data/raw/student_performance.csv').exists() else cwd.parent
data_path = root / 'data/raw/student_performance.csv'
processed_path = root / 'data/processed/cleaned_student_performance.csv'
charts_dir = root / 'outputs/charts'
charts_dir.mkdir(parents=True, exist_ok=True)
print('Project root:', root)

In [ ]:
df = pd.read_csv(data_path)
print(df.shape)
df.head()

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('Duplicate student_id:', df['student_id'].duplicated().sum())

In [ ]:
subject_columns = ['mathematics_marks','science_marks','english_marks','social_science_marks','computer_marks']
# Validation checks
for col in ['attendance_percentage', *subject_columns]:
    invalid = df[~pd.to_numeric(df[col], errors='coerce').between(0,100)]
    print(col, 'invalid/out-of-range:', len(invalid))

# Cleaning: coerce numeric, drop duplicates/invalid rows; do not fill missing marks with 0
clean = df.copy()
clean = clean.drop_duplicates()
clean = clean.drop_duplicates(subset=['student_id'])
for col in ['student_id','attendance_percentage',*subject_columns]:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
clean = clean.dropna(subset=['student_id','student_name','class_name','attendance_percentage',*subject_columns])
clean = clean[clean['attendance_percentage'].between(0,100)]
for col in subject_columns:
    clean = clean[clean[col].between(0,100)]
clean['student_id'] = clean['student_id'].astype(int)
clean['average_score'] = clean[subject_columns].mean(axis=1).round(2)
clean.to_csv(processed_path, index=False)
print('Cleaned rows:', len(clean))

In [ ]:
subject_summary = clean[subject_columns].agg(['mean','median','min','max']).T.round(2)
subject_summary

In [ ]:
def band(score):
    if score >= 80: return '80-100: Excellent'
    if score >= 60: return '60-79.99: Good'
    if score >= 40: return '40-59.99: Needs improvement'
    return 'Below 40: At risk'
clean['performance_band'] = clean['average_score'].apply(band)
band_summary = clean['performance_band'].value_counts().reindex(['80-100: Excellent','60-79.99: Good','40-59.99: Needs improvement','Below 40: At risk'], fill_value=0)
band_summary

In [ ]:
means = clean[subject_columns].mean()
labels = ['Mathematics','Science','English','Social Science','Computer']
plt.figure(figsize=(9,5))
bars = plt.bar(labels, means.values, color='#4C78A8')
plt.title('Subject-wise Average Marks')
plt.ylabel('Average Marks (0-100)')
plt.ylim(0,100)
plt.bar_label(bars, fmt='%.1f')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(charts_dir/'subject_wise_average_marks.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(clean['attendance_percentage'], bins=10, color='#72B7B2', edgecolor='white')
plt.title('Attendance Distribution')
plt.xlabel('Attendance (%)')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.savefig(charts_dir/'attendance_distribution.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(clean['attendance_percentage'], clean['average_score'], alpha=0.75, color='#F58518')
plt.title('Attendance vs Average Marks')
plt.xlabel('Attendance (%)')
plt.ylabel('Average Score (0-100)')
plt.xlim(0,100); plt.ylim(0,100)
plt.tight_layout()
plt.savefig(charts_dir/'attendance_vs_average_marks.png', dpi=150)
plt.show()
corr = clean['attendance_percentage'].corr(clean['average_score'])
print('Correlation (attendance vs average):', round(corr,4))

In [ ]:
plt.figure(figsize=(9,5))
bars = plt.bar(band_summary.index, band_summary.values, color='#54A24B')
plt.title('Student Performance Bands (Project-Defined)')
plt.ylabel('Number of Students')
plt.xticks(rotation=15)
plt.bar_label(bars)
plt.tight_layout()
plt.savefig(charts_dir/'performance_bands.png', dpi=150)
plt.show()

In [ ]:
class_summary = clean.groupby('class_name').agg(avg_score=('average_score','mean'), student_count=('student_id','count')).reset_index()
plt.figure(figsize=(8,5))
bars = plt.bar(class_summary['class_name'], class_summary['avg_score'], color='#E45756')
plt.title('Class-wise Average Performance')
plt.ylabel('Average Score (0-100)')
plt.ylim(0,100)
plt.bar_label(bars, fmt='%.1f')
plt.tight_layout()
plt.savefig(charts_dir/'class_wise_average_performance.png', dpi=150)
plt.show()
class_summary

## Findings (fill after running)

- Highest average subject: Science (71.96)
- Lowest average subject: English (70.62)
- Students below 75% attendance: 16 (26.67%)
- Attendance vs average correlation: 0.1471 (weak positive association; not causation)
- Highest class average: Class 10-A (72.83)
- Data quality: no missing values, duplicates, or out-of-range values in this synthetic dataset
- Limitations: synthetic small dataset (60 rows), project-defined performance bands, correlation does not imply causation